# NBA intro — sportsdataverse-py

ESPN-backed NBA data: play-by-play, schedule, teams, game rosters. Wrappers follow the `espn_nba_*` pattern; pre-built datasets load via `load_nba_*`.

R companion: [hoopR](https://hoopR.sportsdataverse.org). Python neighbor: [nba_api](https://github.com/swar/nba_api) (NBA Stats endpoints). Part of the [SportsDataverse](https://py.sportsdataverse.org/docs/ecosystem).

## Setup

```sh
pip install sportsdataverse
```

In [ ]:
import polars as pl
import sportsdataverse as sdv

## Teams

In [ ]:
teams = sdv.nba.espn_nba_teams()
teams.shape

In [ ]:
teams.select(['team_id', 'team_location', 'team_name', 'team_abbreviation']).head()

## Schedule (ESPN scoreboard, single date)

In [ ]:
schedule = sdv.nba.espn_nba_schedule(dates=20240606)  # 2024 NBA Finals, Game 1
schedule.select(['id', 'home_display_name', 'away_display_name', 'home_score', 'away_score']).head()

## Multi-season schedule via the parquet loader

In [ ]:
schedule_2024 = sdv.nba.load_nba_schedule(seasons=[2024])
schedule_2024.shape

## Play-by-play

`espn_nba_pbp(game_id=...)` returns a dict with the full game payload.

In [ ]:
pbp = sdv.nba.espn_nba_pbp(game_id=401585660)
list(pbp.keys())[:8]

In [ ]:
plays = pl.DataFrame(pbp['plays'], infer_schema_length=None)
plays.select(['period.number', 'clock.displayValue', 'text', 'scoringPlay']).head()

## Game rosters

In [ ]:
rosters = sdv.nba.espn_nba_game_rosters(game_id=401585660)
rosters.select(['athlete_id', 'athlete_display_name', 'team_abbreviation', 'starter']).head()

## Shot-distance distribution (no chart)

Bucket every shot by distance and count attempts. The user can render the result with their preferred plotting library.

In [ ]:
shots = (
    plays
    .filter(pl.col('shootingPlay') == True)
    .with_columns(
        pl.when(pl.col('coordinate.x').is_null()).then(None)
          .otherwise(
              ((pl.col('coordinate.x').cast(pl.Float64, strict=False) ** 2 +
                pl.col('coordinate.y').cast(pl.Float64, strict=False) ** 2) ** 0.5).round(0)
          )
          .alias('shot_distance')
    )
)
shots.select(['period.number', 'text', 'coordinate.x', 'coordinate.y', 'shot_distance']).head()

In [ ]:
(shots
    .with_columns(pl.col('shot_distance').cut([5, 10, 15, 20, 25], labels=['0-5','6-10','11-15','16-20','21-25','25+']).alias('bucket'))
    .group_by('bucket')
    .agg(pl.len().alias('attempts'),
         pl.col('scoringPlay').sum().alias('makes'))
    .sort('bucket'))

## Pipeline example: highest-scoring games of a date range

Pull a multi-day window via `dates=` (range form `YYYYMMDD-YYYYMMDD`).

In [ ]:
window = sdv.nba.espn_nba_schedule(dates='20240606-20240617')  # 2024 Finals window
(window
    .with_columns((pl.col('home_score').cast(pl.Int64, strict=False) + pl.col('away_score').cast(pl.Int64, strict=False)).alias('total'))
    .sort('total', descending=True)
    .select(['date', 'home_display_name', 'away_display_name', 'home_score', 'away_score', 'total'])
    .head())

## Cross-references

- R companion: [hoopR](https://hoopR.sportsdataverse.org)
- Data source: ESPN NBA API
- Stats-API alternative (Python): [nba_api](https://github.com/swar/nba_api)
- Plotting: matplotlib, plotnine

## Where to go next

- API docs: `docs/docs/nba/index.md`
- Next notebook: `05_wbb_wnba_intro.ipynb`